# Gemini Task Analysis Tool
Quickly analyse the performance of a set of Gemini requests and identify any issues.

See go/gemini-task-analysis-overview for a getting started guide.

In [ ]:
#@title Check Imports Available
from colabtools import iterative_builds_client
try:
  from google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis import stubby_samplers
except (ModuleNotFoundError, ImportError):
  iterative_builds_client.build_and_bind(build_targets=['//cloud/ml/aries/orcas_agents/experimental/prompt_critique_agent/task_analysis:task_analysis'])

In [ ]:
# @title Configuration

# @markdown ---
# @markdown ### Project / API details
API_KEY = ''  # @param {type: 'string'}
VERTEX_PROJECT_ID = ''  # @param {type: 'string'}
VERTEX_LOCATION = ''  # @param {type: 'string'}
# @markdown ---

# @markdown ### Input / Output
TASK_NAME = 'Audio Prompts'  # @param {type: 'string'}
INPUT_JSONL_PATH = '/google_src/files/head/depot/google3/cloud/ml/aries/orcas_agents/experimental/prompt_critique_agent/task_analysis/examples/audio_prompts.jsonl'  # @param {type: 'string'}
OUTPUT_HTML_PATH = '' # @param {type: 'string'}

# @markdown ---
# @markdown ### Sampling settings.
# @markdown Model ID should be prefixed 'vertex:' for Vertex or 'gemini:' for Gemini API
MODEL_ID_SAMPLING = 'vertex:gemini-2.5-flash'  # @param {type: 'string'}
NUM_CANDIDATES = 8  # @param {type: 'integer'}
SAMPLING_TEMPERATURE = 1.0  # @param {type: 'number'}
SAMPLING_THINKING_BUDGET = 0 # @param {type: 'number'}
MODEL_ID_CRITIQUE = 'vertex:gemini-2.5-pro'  # @param {type: 'string'}
CRITIQUE_THINKING_BUDGET = -1 # @param {type: 'number'}

# @markdown ---
# @markdown ### CITC Override.
# @markdown If not set, notebook will load coad from current CITC (google3 notebooks) or head (Drive notebooks)
CITC_USER = ''  # @param {type: 'string'}
CITC_WORKSPACE = ''  # @param {type: 'string'}

In [ ]:
# @title Imports and Setup

# Patch to ensure that tqdm progress bars always render in Colab style
from tqdm.notebook import tqdm as notebook_tqdm
import tqdm
import tqdm.std
tqdm.tqdm = notebook_tqdm
tqdm.std.tqdm = notebook_tqdm

import sys
from colabtools import notebookinfo
from etils import ecolab
from IPython import display


def get_import_location():
  if CITC_USER and CITC_WORKSPACE:
    return f'/google_src/cloud/{CITC_USER}/{CITC_WORKSPACE}/google3'
  elif notebookinfo.get_google3_base():
    return notebookinfo.get_google3_base()
  else:
    return '/google_src/head/depot/google3'

with ecolab.adhoc(source=get_import_location(),
                 reload='google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis',
                 cell_autoreload=True,
):
  from google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis import sampling
  from google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis import stubby_samplers
  from google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis import critique_summary
  from google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis import visualization
  from google3.cloud.ml.aries.orcas_agents.experimental.prompt_critique_agent.task_analysis import utils

stubby_samplers.set_gemini_api_key(API_KEY)
stubby_samplers.init_vertex(VERTEX_PROJECT_ID, VERTEX_LOCATION)


In [ ]:
# @title Create Samplers - Generation parameters can be customised

sampler = stubby_samplers.get_sampler(
    MODEL_ID_SAMPLING,
    generation_config={
        'candidate_count': NUM_CANDIDATES,
        'temperature': SAMPLING_TEMPERATURE,
        'thinking_config': {
            'thinking_budget': SAMPLING_THINKING_BUDGET
        }
    },
)
critiquer = stubby_samplers.get_sampler(
    MODEL_ID_CRITIQUE,
    generation_config={
      'temperature': 0.0,
      'thinking_config': {
          'thinking_budget': CRITIQUE_THINKING_BUDGET
      }
    }
)

In [ ]:
# @title Test API Works

request = sampling.prompt_to_request("Write a story about a magic backpack")
response = await sampling.sample(request, sampler)
print(f"✅ Got response with {len(response['candidates'])} candidates.")

In [ ]:
#@title Run critique

requests = utils.read_jsonl(INPUT_JSONL_PATH)
summary = await critique_summary.resample_and_critique(
    TASK_NAME,
    requests,
    resample_sampler=sampler,
    critique_sampler=critiquer)
display.clear_output()

In [ ]:
#@title Display Report
visualization.display_results(summary)

In [ ]:
#@title (Optional) Write report & JSON
visualization.write_results_html(summary, OUTPUT_HTML_PATH)